In [2]:
import sys
import pandas as pd
import numpy as np

In [3]:
movies = pd.read_csv(r"..\data\processed\movies_clean.csv") 
ratings_small = pd.read_csv(r"..\data\raw\ratings_small.csv")
# ratings = pd.read_csv(r"..\data\processed\ratings_clean.csv")

In [4]:
sys.path.append(r"..\src\models")   

from hybrid.hybrid import (
    recommend_hybrid
)

In [5]:
sys.path.append(r"..\src\models")   

from cf.mf_cf import (
    train_mf
)

mf_model = train_mf(ratings_small)

In [6]:
sys.path.append(r"..\src\models")   

from cb.content_based import (
    compute_tf_matrix
)

content_matrix = compute_tf_matrix(movies)
content_index = pd.Index(movies['movieId'])

In [7]:
sys.path.append(r"..\src\models")   

from cb.user_based import (
    build_user_profile
)

movie_indices = pd.Series(movies.index, index=movies['tmdbId'])

In [8]:
tfidf_matrix = compute_tf_matrix(movies)
movie_matrix_dense = tfidf_matrix.toarray()
movie_norms = np.linalg.norm(movie_matrix_dense, axis=1)

In [9]:
# recommendation for user 1
recommendations = recommend_hybrid(user_id=1, movies_df=movies, ratings_df=ratings_small, movie_norms=movie_norms,
                                   mf_model=mf_model,
                                   content_item_matrix=content_matrix,
                                   content_item_index=content_index,
                                   user_profile_vec=build_user_profile(1, ratings_small, content_matrix, content_index),
                                   weights={'mf':0.7, 'content':0.3},
                                   n=10)
recommendations

[2571, 778, 858, 5114, 899, 50, 3730, 2318, 48516, 968]

In [10]:
top_movies_user1 = movies[movies['movieId'].isin(recommendations)][['movieId','original_title','genres','cast','director']]
top_movies_user1 = top_movies_user1.set_index('movieId').loc[recommendations].reset_index()
top_movies_user1

,movieId,original_title,genres,cast,director
0,2571,The Matrix,"['Action', 'Science Fiction']","['Keanu Reeves', 'Laurence Fishburne', 'Carrie...",Lana Wachowski
1,778,Trainspotting,"['Drama', 'Crime']","['Ewan McGregor', 'Ewen Bremner', 'Jonny Lee M...",Danny Boyle
2,858,The Godfather,"['Drama', 'Crime']","['Marlon Brando', 'Al Pacino', 'James Caan']",Francis Ford Coppola
3,5114,The Bad and the Beautiful,"['Drama', 'Romance']","['Lana Turner', 'Kirk Douglas', 'Walter Pidgeon']",Vincente Minnelli
4,899,Singin' in the Rain,"['Comedy', 'Music', 'Romance']","['Gene Kelly', ""Donald O'Connor"", 'Debbie Reyn...",Stanley Donen
5,50,The Usual Suspects,"['Drama', 'Crime', 'Thriller']","['Stephen Baldwin', 'Gabriel Byrne', 'Chazz Pa...",Bryan Singer
6,3730,The Conversation,"['Crime', 'Drama', 'Mystery']","['Gene Hackman', 'John Cazale', 'Frederic Forr...",Francis Ford Coppola
7,2318,Happiness,"['Comedy', 'Drama']","['Jane Adams', 'Jon Lovitz', 'Philip Seymour H...",Todd Solondz
8,48516,The Departed,"['Drama', 'Thriller', 'Crime']","['Leonardo DiCaprio', 'Matt Damon', 'Jack Nich...",Martin Scorsese
9,968,Night of the Living Dead,['Horror'],"['Duane Jones', ""Judith O'Dea"", 'Karl Hardman']",George A. Romero


In [12]:
movies.shape

(43416, 19)

In [13]:
tfidf_matrix.shape

(43416, 5000)